# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL based on the FAIR^2 record.

- [FAIR^2 Croissant schema JSON-LD](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if needed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access metadata fields without iteration or dict-like usage
meta = dataset.metadata
print(meta.name)
print(meta.description)
print(f"Published on: {meta.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Below we enumerate all record sets from the dataset schema, printing their `@id`, title, and the fields within each (by `@id`).

In [ ]:
# Get all record sets by their @id
record_sets = dataset.metadata.recordSet
record_set_ids = []

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    title = rs.get('name', '(no name)')
    print(f"  Title: {title}")
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    @id: {f['@id']}")
    else:
        print("  No fields listed.")
    print()

## 3. Data Extraction
Load data from selected record sets into DataFrames for further analysis.

For demonstration, we'll load the first available record set and display columns with their corresponding field `@id` values.

In [ ]:
# If there are record sets with @id, load them
dataframes = {}
if record_set_ids:
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for record set '{rs_id}': {df.shape[0]} rows, {df.shape[1]} columns.")
            print(f"Columns (@id): {df.columns.tolist()}")
else:
    print("No record sets found in the metadata.")

Below, display the head of the first DataFrame loaded for quick inspection:

In [ ]:
# Display the head for the first record set as example
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"Showing first 5 rows for record set '{first_rs}':")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalizing, and grouping. We reference all fields only by their `@id`.

- Filter numeric fields
- Normalize values
- Group by categorical or demographic attribute

Below, we pick a numeric field and a group field by their `@id`. Please consult the previous overview for available fields.

In [ ]:
# Example: use your own @id values according to the dataset record set and fields
rs_id = list(dataframes.keys())[0]
df = dataframes[rs_id]

# Try to select numeric and group fields by @id -- adjust if necessary
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'loglikelihood' in col.lower():
        numeric_field_id = col
    if 'gender' in col.lower() or 'ward' in col.lower():
        group_field_id = col

if numeric_field_id:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > mean:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    # Group by group field if exists
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found. Please check column names and adjust the script if needed.")

## 5. Visualization
Visualize distributions or relationships for fields referenced by `@id`. We'll plot the distribution of a numeric field and create a group plot if a categorical field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
if numeric_field_id:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group field is available:
    if group_field_id:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field for plotting; check the available columns.")

## 6. Conclusion
This notebook demonstrated how to load a Croissant-encoded dataset, extract records by their `@id`, and conduct exploratory analysis and basic visualization referencing all entities and fields only by their `@id`. The FAIR^2 dataset provides a rich source for understanding gender, socio-economic, and adoption patterns in rangeland management, with limitations noted in missing data and bias. Use the above workflow as a foundation for deeper policy, academic, or community-level analysis.

**Please ensure all field names and transformations follow the `@id` convention for reproducibility.**